In [1]:
# Colab Cell 1: Setup and Environment

# Replace 'Your-Username/Your-Repo-Name' with your actual GitHub path
REPO_URL = 'https://github.com/FatemehRafiei/Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model.git'

REPO_NAME = 'Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model'

# Clone the repository
!git clone {REPO_URL}

# Change the current working directory to the repository's root folder
%cd {REPO_NAME}

Cloning into 'Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 269 (delta 47), reused 0 (delta 0), pack-reused 178 (from 2)
Receiving objects: 100% (269/269), 6.52 MiB | 31.78 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model


In [ ]:
pip install -U arviz

In [6]:
# spatial_sdm.py
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

import pytensor.tensor as pt

# ------------------------------------------------------------
# 0) Define Data Path
# ------------------------------------------------------------
try:
    # 1. Preferred method for scripts (.py files):
    # Get the directory of the current script file.
    BASE_DIR = Path(file).parent
except NameError:
    # 2. Fallback for notebooks (Colab/Jupyter):
    # Use the current working directory, which should be the Git root
    # after cloning and changing directory (see Colab setup below).
    BASE_DIR = Path(os.getcwd())

# Define the relative path to the raw data folder
RAW_DATA_PATH = BASE_DIR / "data"

# Sanity check (Optional, but useful for debugging)
if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"The data/raw directory was not found at the expected path: {RAW_DATA_PATH}"
    )

print(f"Data will be loaded from: {RAW_DATA_PATH}")

Data will be loaded from: /content/Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model/data


In [3]:
import spatial_sdm as sdm

In [ ]:
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

# spatial_sdm.py
"""
Spatial SDM (Spatial Durbin Model) for panel data in PyMC.
...
"""

from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import os


# ------------------------------------------------------------
# 0) Define Data Path
# ------------------------------------------------------------
try:
    # 1. Preferred method for scripts (.py files):
    # Get the directory of the current script file.
    BASE_DIR = Path(__file__).parent
except NameError:
    # 2. Fallback for notebooks (Colab/Jupyter):
    # Use the current working directory, which should be the Git root
    # after cloning and changing directory (see Colab setup below).
    BASE_DIR = Path(os.getcwd())

# Define the relative path to the raw data folder
RAW_DATA_PATH = BASE_DIR / "data"

# Sanity check (Optional, but useful for debugging)
if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"The data/raw directory was not found at the expected path: {RAW_DATA_PATH}"
    )

print(f"Data will be loaded from: {RAW_DATA_PATH}")

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
# ------------------------------------------------------------
# Use the RAW_DATA_PATH variable and the / operator from pathlib
# to construct the full file path.

df_sorted = pd.read_excel(RAW_DATA_PATH / "df_sorted.xlsx")
W_cul04_raw = pd.read_excel(RAW_DATA_PATH / "W_cul04.xlsx")
W_cul06_raw = pd.read_excel(RAW_DATA_PATH / "W_cul06.xlsx")
W_cul05_raw = pd.read_excel(RAW_DATA_PATH / "W_cul05.xlsx")
W_geo_raw = pd.read_excel(RAW_DATA_PATH / "W_geo.xlsx")
W_trade_raw = pd.read_excel(RAW_DATA_PATH / "W_trade.xlsx")


# ------------------------------------------------------------
# 2) Clean spatial weights and align with panel data
# ------------------------------------------------------------
W_cul04 = sdm.prepare_W_from_excel(W_cul04_raw)
W_cul06 = sdm.prepare_W_from_excel(W_cul06_raw)
W_cul05 = sdm.prepare_W_from_excel(W_cul05_raw)
W_geo = sdm.prepare_W_from_excel(W_geo_raw)
W_trade = sdm.prepare_W_from_excel(W_trade_raw)


# ** IMPORTANT: If you want to assign a different weight matrix for this execution,
# you should change the weights matrices below to the desired ones. **
# This can be done by modifying the corresponding W_* variables.

df, W = sdm.align_df_and_W(df_sorted, W_trade)

# Ensure correct sorting (VERY IMPORTANT)
df = df.sort_values(["year", "country"]).copy()
df["year"] = df["year"].astype(int)

# ------------------------------------------------------------
# 3) Sanity checks
# ------------------------------------------------------------
years = sorted(df["year"].unique())
countries = sorted(df["country"].unique())

print("Number of countries (N):", len(countries))
print("Number of years (T):", len(years))
print("Number of observations (NT):", df.shape[0])
print("Expected NT = N * T:", len(countries) * len(years))
print("Balanced panel:", df.shape[0] == len(countries) * len(years))
print("W shape:", W.shape)

# ------------------------------------------------------------
# 4) Fit SDM on the full dataset
# ------------------------------------------------------------



trace_full, countries_sorted, years_sorted, W_base = sdm.run_sdm_model(
    df, W,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=True,
)

print("SDM estimation completed.")

# ------------------------------------------------------------
# 5) (Optional) Save posterior samples for reproducibility
# ------------------------------------------------------------
import arviz as az
az.to_netcdf(trace_full, "trace_sdm_full.nc")


Data will be loaded from: /content/Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model/data
Number of countries (N): 11
Number of years (T): 18
Number of observations (NT): 198
Expected NT = N * T: 198
Balanced panel: True
W shape: (11, 11)


Output()

In [ ]:
import importlib
import spatial_sdm as sdm
importlib.reload(sdm)

summary_main, df_alpha, df_time_alpha = sdm.summarize_sdm_trace(
    trace_full, countries_sorted, years_sorted, round_to=4, verbose=True
)

summary_main
df_alpha.head()
df_time_alpha.head()


In [ ]:
import numpy as np
import pandas as pd
#import arviz as az
import spatial_sdm as sdm

X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T = sdm.build_design_mats(df, W_base)

y_hat = sdm.posterior_mean_predictions(trace_full, X, WX, country_idx, year_idx, W_base, N=N, T=T)

mse_by_country = {}
for c_i, c in enumerate(countries_sorted):
    mask = (country_idx == c_i)
    mse_by_country[c] = float(np.mean((y[mask] - y_hat[mask])**2))

mse_df = pd.DataFrame({"country": list(mse_by_country.keys()),
                       "mse": list(mse_by_country.values())}).sort_values("mse")

print("MSE by country (lower is better):")
display(mse_df)


# Optional: save
mse_df.to_csv("mse_by_country.csv", index=False)


# ------------------------------------------------------------
# 3) Moran's I on SDM residuals by year
# ------------------------------------------------------------
# We need a PySAL weights object + residuals per year
from libpysal.weights import W as W_pysal
from esda.moran import Moran


morans_df = sdm.morans_I_by_year(
    trace=trace_full,
    X=X,
    WX=WX,
    y=y,
    country_idx=country_idx,
    year_idx=year_idx,
    W_base=W_base,
    years_sorted=years_sorted,
    N=N,
    T=T,
)

display(morans_df)
morans_df.to_csv("moransI_residuals_by_year.csv", index=False)


In [ ]:
# ------------------------------------------------------------
# Leave-One-Country-Out (LOCO) cross-validation
# ------------------------------------------------------------
# NOTE: This is computationally expensive.
# Start with small draws/tune for testing.

res_loco = sdm.loco_cv(
    df=df,
    W_df=W,
    draws=1000,           # increase after testing
    tune=1000,
    chains=2,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=False,
)

print(res_loco["status"].value_counts())
print(res_loco.head())

# Save LOCO results
res_loco.to_csv("loco_results.csv", index=False)

print("LOCO cross-validation completed.")
